# Multi-Coin LSTM Training Pipeline with MLflow Integration

This notebook implements a complete training pipeline for binary classification of trading signals across multiple cryptocurrencies.

## Pipeline Overview:
1. **Data Building**: Create binary indicator dataframes for each coin using DatasetBuilder (grid search optimization)
2. **Preprocessing**: Align and preprocess data using DataPreprocessor
3. **Sequence Creation**: Create sliding window sequences for LSTM input
4. **Multi-Coin Concatenation**: Combine all coins into unified train/val/test datasets
5. **Model Training**: Train CNNLSTMSignalPredictor with MLflow tracking
6. **Evaluation**: Comprehensive metrics and visualizations

## Section 1: Setup and Configuration

In [ ]:
# Cell 1: Imports and Path Setup
import sys
from pathlib import Path
import json
import os
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Any
from dataclasses import dataclass, asdict, field
import pickle

# Add parent directory for imports
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

# LSTM modules
from crypto_analysis.lstm.model import ModelConfig, CNNLSTMSignalPredictor
from crypto_analysis.lstm.trainer import Trainer, TrainingConfig, TrainingHistory
from crypto_analysis.lstm.loss import BinarySignalLoss, FocalBinaryLoss
from crypto_analysis.lstm.data_preprocessor import DataPreprocessor
from crypto_analysis.lstm.dataset import SignalDataset, create_sequences

# Dataset builder
from crypto_analysis.indicator_optimizer.dataset_builder import DatasetBuilder

# MLflow
import mlflow
import mlflow.pytorch

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
print(f"MLflow version: {mlflow.__version__}")

In [ ]:
# Cell 2: Configuration Constants

# === CONFIGURATION ===

# Data paths
DATA_DIR = Path("../data/binance")
OUTPUT_DIR = Path("coin_csvs")
OUTPUT_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / "checkpoints").mkdir(exist_ok=True)

# Available symbols (actual files in data directory)
AVAILABLE_SYMBOLS = [
    "ADA", "ATOM", "AVAX", "BNB", "BTC", "DOGE", "ETH", "ICP",
    "LINK", "NEAR", "OP", "PEPE", "SHIB", "SOL", "XLM", "XRP"
]

# DatasetBuilder parameters
DATASET_CONFIG = {
    "period_hours": 4,
    "signal_shift": 0,  # No shifting in DatasetBuilder (indicator signals align with current time)
    "threshold_pct": 1.2,
    "grid_search": True,
    "hyperopt": False,
}

# Sequence parameters
SEQUENCE_CONFIG = {
    "input_seq_length": 36,  # 36 timesteps lookback
    "output_seq_length": 1,  # Single binary prediction
    "stride": 4,  # Period-aligned stride
}

# Preprocessor parameters
PREPROCESSOR_CONFIG = {
    "period_size": 4,  # For alignment
    "target_shift": 4,  # Features at t predict target at t+4
}

# Train/val/test split ratios
SPLIT_CONFIG = {
    "train_ratio": 0.6,
    "val_ratio": 0.2,
    "test_ratio": 0.2,
}

# MLflow experiment name
MLFLOW_EXPERIMENT = "multi_coin_lstm_training"

print("Configuration loaded.")
print(f"Data directory: {DATA_DIR.absolute()}")
print(f"Output directory: {OUTPUT_DIR.absolute()}")

In [ ]:
# Cell 3: Symbol Validation Utility

def get_available_symbols(data_dir: Path, requested_symbols: List[str]) -> List[str]:
    """
    Filter requested symbols to only those with available data files.
    
    Parameters
    ----------
    data_dir : Path
        Directory containing feather files
    requested_symbols : List[str]
        List of symbol names (e.g., ["BTC", "ETH"])
    
    Returns
    -------
    List[str]
        Filtered list of symbols with available data
    """
    available = []
    missing = []
    
    for symbol in requested_symbols:
        # Check for any timeframe file
        pattern = f"{symbol}_USDT-*.feather"
        files = list(data_dir.glob(pattern))
        if files:
            available.append(symbol)
        else:
            missing.append(symbol)
    
    if missing:
        print(f"Warning: Data files not found for: {missing}")
    
    return available

# Validate symbols
symbols = get_available_symbols(DATA_DIR, AVAILABLE_SYMBOLS)
print(f"\nAvailable symbols ({len(symbols)}): {symbols}")

## Section 2: Data Building for All Coins

In [ ]:
# Cell 4: Data Building Utility Functions

def build_coin_dataframes(
    symbols: List[str],
    data_dir: Path,
    config: Dict,
    verbose: bool = True
) -> Tuple[Dict[str, pd.DataFrame], Dict[str, Dict]]:
    """
    Build binary indicator dataframes for all coins using DatasetBuilder.
    
    Parameters
    ----------
    symbols : List[str]
        List of coin symbols
    data_dir : Path
        Directory with data files
    config : Dict
        DatasetBuilder configuration (period_hours, signal_shift, threshold_pct, etc.)
    verbose : bool
        Print progress
    
    Returns
    -------
    Tuple[Dict[str, pd.DataFrame], Dict[str, Dict]]
        (dataframes, optimization_results) mappings
    """
    builder = DatasetBuilder(
        data_dir=data_dir,
        period_hours=config["period_hours"],
        signal_shift=config["signal_shift"],
        output_mode='binary'
    )
    
    dataframes = {}
    optimization_results = {}
    
    for i, symbol in enumerate(symbols):
        if verbose:
            print(f"\n[{i+1}/{len(symbols)}] Building dataset for {symbol}...")
        
        try:
            df = builder.build(
                symbol=symbol,
                threshold_pct=config["threshold_pct"],
                grid_search=config["grid_search"],
                hyperopt=config["hyperopt"],
                verbose=False  # Reduce noise
            )
            dataframes[symbol] = df
            optimization_results[symbol] = builder.get_optimization_results(symbol)
            
            if verbose:
                print(f"  Shape: {df.shape}")
                trade_count = (df['tradeable'] == 'trade').sum()
                hold_count = (df['tradeable'] == 'hold').sum()
                trade_pct = trade_count / len(df) * 100
                print(f"  Trade: {trade_count} ({trade_pct:.2f}%) | Hold: {hold_count}")
                
        except Exception as e:
            print(f"  Error: {e}")
    
    return dataframes, optimization_results


def save_dataframes_to_csv(
    dataframes: Dict[str, pd.DataFrame],
    output_dir: Path
) -> Dict[str, Path]:
    """
    Save each coin's dataframe as CSV.
    
    Parameters
    ----------
    dataframes : Dict[str, pd.DataFrame]
        Symbol -> DataFrame mapping
    output_dir : Path
        Output directory
    
    Returns
    -------
    Dict[str, Path]
        Symbol -> CSV path mapping
    """
    paths = {}
    for symbol, df in dataframes.items():
        csv_path = output_dir / f"{symbol.lower()}_binary.csv"
        df.to_csv(csv_path, index=False)
        paths[symbol] = csv_path
        print(f"Saved {symbol}: {csv_path}")
    return paths


def save_optimization_params(
    optimization_results: Dict[str, Dict],
    output_dir: Path
) -> Path:
    """
    Save optimization parameters for all coins to JSON.
    
    Parameters
    ----------
    optimization_results : Dict[str, Dict]
        Symbol -> {indicator_name: OptimizationResult} mapping
    output_dir : Path
        Output directory
    
    Returns
    -------
    Path
        Path to saved JSON file
    """
    params_dict = {}
    for symbol, results in optimization_results.items():
        params_dict[symbol] = {}
        for ind_name, result in results.items():
            params_dict[symbol][ind_name] = {
                "indicator_name": result.indicator_name,
                "optimization_type": result.optimization_type,
                "score": float(result.score),
                "best_params": result.best_params
            }
    
    output_path = output_dir / "optimization_params.json"
    with open(output_path, 'w') as f:
        json.dump(params_dict, f, indent=2)
    
    print(f"\nSaved optimization params: {output_path}")
    return output_path


def print_data_summary(dataframes: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Print summary statistics for all coin dataframes."""
    print("=" * 80)
    print("DATA SUMMARY")
    print("=" * 80)
    
    summary_data = []
    metadata_cols = ['date', 'signal', 'signal_pct_change', 'period_id', 'tradeable']
    
    for symbol, df in dataframes.items():
        n_rows = len(df)
        n_features = len([c for c in df.columns if c not in metadata_cols])
        trade_count = (df['tradeable'] == 'trade').sum()
        hold_count = (df['tradeable'] == 'hold').sum()
        trade_pct = trade_count / n_rows * 100
        
        summary_data.append({
            'Symbol': symbol,
            'Rows': n_rows,
            'Features': n_features,
            'Trade': trade_count,
            'Hold': hold_count,
            'Trade %': f"{trade_pct:.2f}%"
        })
    
    summary_df = pd.DataFrame(summary_data)
    print(summary_df.to_string(index=False))
    print("=" * 80)
    return summary_df

In [ ]:
# Cell 5: Build all coin dataframes
# This may take several minutes due to grid search optimization

print("Building binary indicator dataframes for all coins...")
print("This process includes grid search optimization for each indicator.")
print("-" * 60)

coin_dataframes, optimization_results = build_coin_dataframes(
    symbols, DATA_DIR, DATASET_CONFIG, verbose=True
)

print(f"\nSuccessfully built {len(coin_dataframes)} dataframes.")

In [ ]:
# Cell 6: Save dataframes and optimization params

print("Saving dataframes to CSV...")
csv_paths = save_dataframes_to_csv(coin_dataframes, OUTPUT_DIR)

print("\nSaving optimization parameters...")
params_path = save_optimization_params(optimization_results, OUTPUT_DIR)

In [ ]:
# Cell 7: Data summary

data_summary = print_data_summary(coin_dataframes)

### Alternative: Load Pre-built Data from CSV

If you've already run `build_coin_data.py` (multiprocessing script) or the data building cells above, you can skip the data building step and load directly from saved CSV files.

In [ ]:
# Cell: Load Pre-built Data from CSV (Alternative to Data Building)
# Run this cell INSTEAD of cells 5-7 if you've already built the data

def load_coin_dataframes_from_csv(
    output_dir: Path,
    symbols: List[str],
    verbose: bool = True
) -> Tuple[Dict[str, pd.DataFrame], Dict[str, Dict]]:
    """
    Load pre-built dataframes and optimization params from CSV files.
    
    This provides the same data structure as build_coin_dataframes() so you can
    skip the slow data building step if CSVs already exist.
    
    Parameters
    ----------
    output_dir : Path
        Directory containing CSV files and optimization_params.json
    symbols : List[str]
        List of symbols to load (will filter to those with available CSVs)
    verbose : bool
        Print progress
    
    Returns
    -------
    Tuple[Dict[str, pd.DataFrame], Dict[str, Dict]]
        (coin_dataframes, optimization_results) - same structure as build_coin_dataframes()
    """
    coin_dataframes = {}
    loaded_symbols = []
    missing_symbols = []
    
    if verbose:
        print("Loading pre-built dataframes from CSV files...")
        print("-" * 60)
    
    for symbol in symbols:
        csv_path = output_dir / f"{symbol.lower()}_binary.csv"
        if csv_path.exists():
            df = pd.read_csv(csv_path)
            # Parse date column if present
            if 'date' in df.columns:
                df['date'] = pd.to_datetime(df['date'])
            coin_dataframes[symbol] = df
            loaded_symbols.append(symbol)
            
            if verbose:
                trade_count = (df['tradeable'] == 'trade').sum()
                hold_count = (df['tradeable'] == 'hold').sum()
                trade_pct = trade_count / len(df) * 100
                print(f"  {symbol}: {df.shape} | Trade: {trade_count} ({trade_pct:.1f}%)")
        else:
            missing_symbols.append(symbol)
    
    if verbose:
        print("-" * 60)
        print(f"Loaded {len(loaded_symbols)} dataframes: {loaded_symbols}")
        if missing_symbols:
            print(f"Missing CSVs for: {missing_symbols}")
    
    # Load optimization params
    optimization_results = {}
    params_path = output_dir / "optimization_params.json"
    
    if params_path.exists():
        with open(params_path, 'r') as f:
            optimization_results = json.load(f)
        if verbose:
            print(f"\nLoaded optimization params from: {params_path}")
    else:
        if verbose:
            print(f"\nWarning: optimization_params.json not found at {params_path}")
    
    return coin_dataframes, optimization_results


# === UNCOMMENT AND RUN THIS TO LOAD FROM CSV ===
# coin_dataframes, optimization_results = load_coin_dataframes_from_csv(
#     OUTPUT_DIR, symbols, verbose=True
# )
# 
# # Print data summary
# data_summary = print_data_summary(coin_dataframes)
# 
# # Update symbols to only those that were loaded
# symbols = list(coin_dataframes.keys())
# print(f"\nActive symbols ({len(symbols)}): {symbols}")

## Section 3: Data Preprocessing

In [ ]:
# Cell 8: Preprocessing Utility Functions

def preprocess_coin_dataframes(
    dataframes: Dict[str, pd.DataFrame],
    period_size: int = 4,
    target_shift: int = 4,
    verbose: bool = True
) -> Dict[str, Tuple[np.ndarray, np.ndarray, DataPreprocessor]]:
    """
    Preprocess each coin's dataframe using DataPreprocessor.
    
    Parameters
    ----------
    dataframes : Dict[str, pd.DataFrame]
        Symbol -> DataFrame mapping
    period_size : int
        Period size for alignment
    target_shift : int
        Target shift for preprocessor (features at t predict t+shift)
    verbose : bool
        Print progress
    
    Returns
    -------
    Dict[str, Tuple[np.ndarray, np.ndarray, DataPreprocessor]]
        Symbol -> (features, targets, preprocessor) mapping
    """
    processed = {}
    
    for symbol, df in dataframes.items():
        if verbose:
            print(f"\nPreprocessing {symbol}...")
        
        # Align dataframe for period consistency
        df_aligned = DataPreprocessor.align_dataframe(
            df, period_size=period_size, verbose=verbose
        )
        
        # Initialize and fit preprocessor
        preprocessor = DataPreprocessor(target_shift=target_shift)
        features, targets = preprocessor.fit_transform(df_aligned)
        
        processed[symbol] = (features, targets, preprocessor)
        
        if verbose:
            print(f"  Features shape: {features.shape}")
            print(f"  Targets shape: {targets.shape}")
            unique, counts = np.unique(targets, return_counts=True)
            dist = dict(zip(['hold', 'trade'], counts))
            print(f"  Target distribution: {dist}")
    
    return processed


def get_unified_feature_info(
    processed_data: Dict[str, Tuple[np.ndarray, np.ndarray, DataPreprocessor]]
) -> Tuple[int, List[str]]:
    """
    Verify all preprocessors have same feature configuration.
    
    Parameters
    ----------
    processed_data : Dict
        Preprocessed data from preprocess_coin_dataframes
    
    Returns
    -------
    Tuple[int, List[str]]
        (n_features, feature_names)
    """
    feature_counts = {}
    feature_names = None
    
    for symbol, (features, targets, preprocessor) in processed_data.items():
        n_features = preprocessor.get_num_features()
        names = preprocessor.get_feature_names()
        feature_counts[symbol] = n_features
        if feature_names is None:
            feature_names = names
    
    # Check all have same feature count
    unique_counts = set(feature_counts.values())
    if len(unique_counts) > 1:
        print("Warning: Different feature counts across coins!")
        for symbol, count in feature_counts.items():
            print(f"  {symbol}: {count}")
    else:
        print(f"All coins have {list(unique_counts)[0]} features.")
    
    n_features = list(feature_counts.values())[0]
    print(f"\nFeature names (first 10): {feature_names[:10]}...")
    
    return n_features, feature_names

In [ ]:
# Cell 9: Preprocess all coin dataframes

print("Preprocessing coin dataframes...")
print(f"Period size: {PREPROCESSOR_CONFIG['period_size']}")
print(f"Target shift: {PREPROCESSOR_CONFIG['target_shift']}")
print("-" * 60)

processed_data = preprocess_coin_dataframes(
    coin_dataframes,
    period_size=PREPROCESSOR_CONFIG['period_size'],
    target_shift=PREPROCESSOR_CONFIG['target_shift'],
    verbose=True
)

In [ ]:
# Cell 10: Get unified feature configuration

n_features, feature_names = get_unified_feature_info(processed_data)

## Section 4: Sequence Creation and Validation

In [ ]:
# Cell 11: Sequence Creation Utility Functions

def create_coin_sequences(
    processed_data: Dict[str, Tuple[np.ndarray, np.ndarray, DataPreprocessor]],
    seq_config: Dict,
    verbose: bool = True
) -> Dict[str, Tuple[np.ndarray, np.ndarray]]:
    """
    Create sequences for each coin.
    
    Parameters
    ----------
    processed_data : Dict
        Preprocessed data
    seq_config : Dict
        Sequence configuration (input_seq_length, output_seq_length, stride)
    verbose : bool
        Print progress
    
    Returns
    -------
    Dict[str, Tuple[np.ndarray, np.ndarray]]
        Symbol -> (feature_sequences, target_sequences) mapping
    """
    sequences = {}
    
    print(f"Creating sequences with:")
    print(f"  input_seq_length: {seq_config['input_seq_length']}")
    print(f"  output_seq_length: {seq_config['output_seq_length']}")
    print(f"  stride: {seq_config['stride']}")
    print("-" * 60)
    
    for symbol, (features, targets, _) in processed_data.items():
        try:
            feat_seqs, tgt_seqs = create_sequences(
                features, targets,
                input_seq_length=seq_config["input_seq_length"],
                output_seq_length=seq_config["output_seq_length"],
                stride=seq_config["stride"]
            )
            sequences[symbol] = (feat_seqs, tgt_seqs)
            
            if verbose:
                print(f"{symbol}: features {feat_seqs.shape}, targets {tgt_seqs.shape}")
        except ValueError as e:
            print(f"{symbol}: Error - {e}")
    
    return sequences


def validate_target_sequences(
    sequences: Dict[str, Tuple[np.ndarray, np.ndarray]]
) -> Dict[str, Dict]:
    """
    Validate that target sequences have no mixed labels.
    For output_seq_length=1, targets are scalars (always valid).
    
    Parameters
    ----------
    sequences : Dict[str, Tuple[np.ndarray, np.ndarray]]
        Symbol -> (feature_seqs, target_seqs) mapping
    
    Returns
    -------
    Dict[str, Dict]
        Validation results per symbol
    """
    print("\n" + "=" * 70)
    print("TARGET SEQUENCE VALIDATION")
    print("=" * 70)
    
    results = {}
    
    for symbol, (feat_seqs, tgt_seqs) in sequences.items():
        unique_targets = np.unique(tgt_seqs)
        hold_count = int((tgt_seqs == 0).sum())
        trade_count = int((tgt_seqs == 1).sum())
        total = len(tgt_seqs)
        
        # Check for mixed labels (only applicable if output_seq_length > 1)
        # For output_seq_length=1, targets are scalars, so no mixing possible
        is_valid = len(unique_targets) <= 2 and all(u in [0, 1] for u in unique_targets)
        
        results[symbol] = {
            "total_sequences": total,
            "hold_count": hold_count,
            "trade_count": trade_count,
            "hold_pct": hold_count / total * 100,
            "trade_pct": trade_count / total * 100,
            "unique_labels": unique_targets.tolist(),
            "valid": is_valid
        }
        
        status = "OK" if is_valid else "INVALID"
        print(f"{symbol}: {total:5d} seqs | Hold: {hold_count:5d} ({results[symbol]['hold_pct']:5.1f}%) | "
              f"Trade: {trade_count:5d} ({results[symbol]['trade_pct']:5.1f}%) | [{status}]")
    
    print("=" * 70)
    
    # Summary
    all_valid = all(r["valid"] for r in results.values())
    if all_valid:
        print("All sequences validated successfully - no mixed labels.")
    else:
        invalid = [s for s, r in results.items() if not r["valid"]]
        print(f"WARNING: Invalid sequences found for: {invalid}")
    
    return results

In [ ]:
# Cell 12: Create sequences for all coins

coin_sequences = create_coin_sequences(processed_data, SEQUENCE_CONFIG, verbose=True)

In [ ]:
# Cell 13: Validate target sequences

sequence_validation = validate_target_sequences(coin_sequences)

## Section 5: Multi-Coin Dataset Concatenation

In [ ]:
# Cell 14: Dataset Splitting and Concatenation Utilities

def split_coin_sequences(
    sequences: Dict[str, Tuple[np.ndarray, np.ndarray]],
    split_config: Dict
) -> Dict[str, Dict[str, Tuple[np.ndarray, np.ndarray]]]:
    """
    Split each coin's sequences into train/val/test sets.
    Uses sequential split (no shuffle) to preserve temporal order.
    
    Parameters
    ----------
    sequences : Dict[str, Tuple[np.ndarray, np.ndarray]]
        Symbol -> (feature_seqs, target_seqs) mapping
    split_config : Dict
        Split ratios (train_ratio, val_ratio, test_ratio)
    
    Returns
    -------
    Dict[str, Dict[str, Tuple[np.ndarray, np.ndarray]]]
        Symbol -> {'train': (feat, tgt), 'val': (feat, tgt), 'test': (feat, tgt)}
    """
    split_data = {}
    
    train_ratio = split_config["train_ratio"]
    val_ratio = split_config["val_ratio"]
    
    print(f"Splitting sequences: {train_ratio*100:.0f}% train, {val_ratio*100:.0f}% val, "
          f"{split_config['test_ratio']*100:.0f}% test")
    print("-" * 70)
    
    for symbol, (feat_seqs, tgt_seqs) in sequences.items():
        n = len(feat_seqs)
        train_end = int(n * train_ratio)
        val_end = int(n * (train_ratio + val_ratio))
        
        split_data[symbol] = {
            'train': (feat_seqs[:train_end], tgt_seqs[:train_end]),
            'val': (feat_seqs[train_end:val_end], tgt_seqs[train_end:val_end]),
            'test': (feat_seqs[val_end:], tgt_seqs[val_end:])
        }
        
        print(f"{symbol}: train={train_end}, val={val_end - train_end}, test={n - val_end}")
    
    return split_data


def concatenate_coin_datasets(
    split_data: Dict[str, Dict[str, Tuple[np.ndarray, np.ndarray]]],
    device: Optional[torch.device] = None
) -> Dict[str, SignalDataset]:
    """
    Concatenate all coins' train/val/test sets into unified datasets.
    
    Parameters
    ----------
    split_data : Dict
        Split data from split_coin_sequences
    device : torch.device, optional
        Device for tensors
    
    Returns
    -------
    Dict[str, SignalDataset]
        {'train': SignalDataset, 'val': SignalDataset, 'test': SignalDataset}
    """
    combined = {
        'train': {'feat': [], 'tgt': []},
        'val': {'feat': [], 'tgt': []},
        'test': {'feat': [], 'tgt': []}
    }
    
    for symbol, splits in split_data.items():
        for split_name in ['train', 'val', 'test']:
            feat, tgt = splits[split_name]
            combined[split_name]['feat'].append(feat)
            combined[split_name]['tgt'].append(tgt)
    
    datasets = {}
    print("\n" + "=" * 50)
    print("CONCATENATED DATASETS")
    print("=" * 50)
    
    for split_name in ['train', 'val', 'test']:
        feat_concat = np.concatenate(combined[split_name]['feat'], axis=0)
        tgt_concat = np.concatenate(combined[split_name]['tgt'], axis=0)
        datasets[split_name] = SignalDataset(feat_concat, tgt_concat, device=device)
        
        dist = datasets[split_name].get_class_distribution()
        print(f"{split_name:5s}: {len(datasets[split_name]):6d} samples | "
              f"Hold: {dist['hold']:5d} | Trade: {dist['trade']:5d}")
    
    print("=" * 50)
    return datasets


def analyze_dataset_distribution(
    datasets: Dict[str, SignalDataset]
) -> Dict[str, Dict]:
    """
    Analyze class distribution in each dataset split.
    
    Parameters
    ----------
    datasets : Dict[str, SignalDataset]
        Train/val/test datasets
    
    Returns
    -------
    Dict[str, Dict]
        Distribution statistics for each split
    """
    distributions = {}
    
    print("\n" + "=" * 70)
    print("DATASET DISTRIBUTION ANALYSIS")
    print("=" * 70)
    
    for split_name, dataset in datasets.items():
        dist = dataset.get_class_distribution()
        total = dist['hold'] + dist['trade']
        hold_pct = dist['hold'] / total * 100
        trade_pct = dist['trade'] / total * 100
        imbalance_ratio = dist['hold'] / max(dist['trade'], 1)
        
        distributions[split_name] = {
            'total': total,
            'hold': dist['hold'],
            'trade': dist['trade'],
            'hold_pct': hold_pct,
            'trade_pct': trade_pct,
            'imbalance_ratio': imbalance_ratio
        }
        
        print(f"\n{split_name.upper()}:")
        print(f"  Total samples: {total}")
        print(f"  Hold:  {dist['hold']:6d} ({hold_pct:5.2f}%)")
        print(f"  Trade: {dist['trade']:6d} ({trade_pct:5.2f}%)")
        print(f"  Imbalance ratio (hold/trade): {imbalance_ratio:.2f}")
    
    print("\n" + "=" * 70)
    return distributions

In [ ]:
# Cell 15: Split and concatenate datasets

# Split each coin's sequences
split_coin_data = split_coin_sequences(coin_sequences, SPLIT_CONFIG)

# Concatenate into unified datasets
multi_coin_datasets = concatenate_coin_datasets(split_coin_data)

In [ ]:
# Cell 16: Analyze dataset distribution

dataset_distributions = analyze_dataset_distribution(multi_coin_datasets)

## Section 6: Model Configuration

In [ ]:
# Cell 17: Create Model Configuration

model_config = ModelConfig(
    input_size=n_features,
    hidden_size=128,
    num_layers=2,
    dropout=0.2,
    bidirectional=False,
    num_classes=2,
    input_seq_length=SEQUENCE_CONFIG["input_seq_length"],
    classifier_hidden_size=64,
    # CNN-LSTM specific
    kernel_size=3,
    cnn_num_layers=3,
    cnn_dropout=0.1,
    lstm_dropout=0.1,
    classifier_dropout=0.2,
)

print("Model Configuration:")
print("=" * 40)
for field_name, value in asdict(model_config).items():
    print(f"  {field_name}: {value}")

In [ ]:
# Cell 18: Create Model

model = CNNLSTMSignalPredictor(model_config)
print(model)
print(f"\nTotal trainable parameters: {model.get_num_parameters():,}")

## Section 7: Training Configuration with Helper Function

In [ ]:
# Cell 19: Helper Function for Loss Configuration

def configure_loss_settings(
    distributions: Dict[str, Dict],
    verbose: bool = True
) -> Dict:
    """
    Determine optimal loss function settings based on dataset distribution.
    
    Logic:
    - Extreme imbalance (ratio > 10): Use focal loss with high gamma, full class weights
    - High imbalance (ratio 5-10): Use focal loss with moderate gamma
    - Moderate imbalance (ratio 2-5): Use class weights only
    - Balanced (ratio < 2): Minimal adjustments
    
    Parameters
    ----------
    distributions : Dict[str, Dict]
        Distribution statistics from analyze_dataset_distribution
    verbose : bool
        Print reasoning
    
    Returns
    -------
    Dict
        Configuration dict with:
        - auto_class_weights: bool
        - class_weight_power: float (0.0-1.0)
        - focal_loss: bool
        - focal_gamma: float (0.0-5.0)
        - label_smoothing: float (0.0-0.2)
    """
    train_dist = distributions['train']
    imbalance_ratio = train_dist['imbalance_ratio']
    trade_pct = train_dist['trade_pct']
    
    if verbose:
        print("=" * 60)
        print("LOSS CONFIGURATION HELPER")
        print("=" * 60)
        print(f"Train imbalance ratio (hold/trade): {imbalance_ratio:.2f}")
        print(f"Trade percentage: {trade_pct:.2f}%")
    
    # Determine settings based on imbalance
    if imbalance_ratio > 10:
        config = {
            'auto_class_weights': True,
            'class_weight_power': 1.0,
            'focal_loss': True,
            'focal_gamma': 3.0,
            'label_smoothing': 0.1,
        }
        reason = "EXTREME imbalance (>10x) - Using focal loss with high gamma"
        
    elif imbalance_ratio > 5:
        config = {
            'auto_class_weights': True,
            'class_weight_power': 0.75,
            'focal_loss': True,
            'focal_gamma': 2.0,
            'label_smoothing': 0.1,
        }
        reason = "HIGH imbalance (5-10x) - Using focal loss with moderate settings"
        
    elif imbalance_ratio > 2:
        config = {
            'auto_class_weights': True,
            'class_weight_power': 0.5,
            'focal_loss': False,
            'focal_gamma': 2.0,
            'label_smoothing': 0.05,
        }
        reason = "MODERATE imbalance (2-5x) - Using class weights only"
        
    else:
        config = {
            'auto_class_weights': True,
            'class_weight_power': 0.25,
            'focal_loss': False,
            'focal_gamma': 2.0,
            'label_smoothing': 0.0,
        }
        reason = "BALANCED dataset (<2x) - Minimal adjustments"
    
    if verbose:
        print(f"\nDecision: {reason}")
        print(f"\nConfiguration:")
        for key, value in config.items():
            print(f"  {key}: {value}")
        print("=" * 60)
    
    return config

In [ ]:
# Cell 20: Configure loss settings based on distribution

loss_config = configure_loss_settings(dataset_distributions, verbose=True)

In [ ]:
# Cell 21: Create Training Configuration

training_config = TrainingConfig(
    # Model architecture (for reference)
    hidden_size=model_config.hidden_size,
    num_layers=model_config.num_layers,
    dropout=model_config.dropout,
    
    # Training parameters
    epochs=100,
    batch_size=64,
    learning_rate=1e-3,
    weight_decay=1e-3,
    optimizer='adamw',
    grad_clip_norm=1.0,
    
    # Learning rate scheduler
    scheduler='plateau',
    scheduler_patience=10,
    scheduler_factor=0.5,
    
    # Class imbalance handling (from helper function)
    auto_class_weights=loss_config['auto_class_weights'],
    class_weight_power=loss_config['class_weight_power'],
    focal_loss=loss_config['focal_loss'],
    focal_gamma=loss_config['focal_gamma'],
    label_smoothing=loss_config['label_smoothing'],
    
    # Data split (we provide pre-split datasets)
    val_split=0.0,
    test_split=0.0,
    
    # Early stopping
    early_stopping=True,
    patience=20,
    min_delta=1e-4,
    
    # Checkpointing
    checkpoint_dir=str(OUTPUT_DIR / "checkpoints"),
    save_best_only=True,
    
    # Device
    device='auto',
    
    # Logging
    log_interval=20,
    verbose=True,
)

print("Training Configuration created.")
print(f"Epochs: {training_config.epochs}")
print(f"Batch size: {training_config.batch_size}")
print(f"Learning rate: {training_config.learning_rate}")
print(f"Focal loss: {training_config.focal_loss}")
print(f"Class weight power: {training_config.class_weight_power}")

## Section 8: MLflow Integration

In [ ]:
# Cell 22: MLflow Setup and Logging Functions

def setup_mlflow(experiment_name: str) -> str:
    """
    Setup MLflow experiment.
    
    Parameters
    ----------
    experiment_name : str
        Name for the MLflow experiment
    
    Returns
    -------
    str
        Experiment ID
    """
    # Set tracking URI (default is local ./mlruns)
    mlflow.set_tracking_uri("mlruns")
    
    # Create or get experiment
    experiment = mlflow.get_experiment_by_name(experiment_name)
    if experiment is None:
        experiment_id = mlflow.create_experiment(experiment_name)
    else:
        experiment_id = experiment.experiment_id
    
    mlflow.set_experiment(experiment_name)
    
    print(f"MLflow experiment: {experiment_name}")
    print(f"Experiment ID: {experiment_id}")
    print(f"Tracking URI: {mlflow.get_tracking_uri()}")
    
    return experiment_id


def log_training_params(
    model_config: ModelConfig,
    training_config: TrainingConfig,
    dataset_config: Dict,
    sequence_config: Dict,
    symbols: List[str],
    distributions: Dict[str, Dict]
):
    """
    Log all configuration parameters to MLflow.
    """
    # Model params
    mlflow.log_params({
        "model.input_size": model_config.input_size,
        "model.hidden_size": model_config.hidden_size,
        "model.num_layers": model_config.num_layers,
        "model.dropout": model_config.dropout,
        "model.bidirectional": model_config.bidirectional,
        "model.classifier_hidden_size": model_config.classifier_hidden_size,
        "model.kernel_size": model_config.kernel_size,
        "model.cnn_num_layers": model_config.cnn_num_layers,
        "model.cnn_dropout": model_config.cnn_dropout,
        "model.lstm_dropout": model_config.lstm_dropout,
    })
    
    # Training params
    mlflow.log_params({
        "train.epochs": training_config.epochs,
        "train.batch_size": training_config.batch_size,
        "train.learning_rate": training_config.learning_rate,
        "train.weight_decay": training_config.weight_decay,
        "train.optimizer": training_config.optimizer,
        "train.scheduler": training_config.scheduler,
        "train.focal_loss": training_config.focal_loss,
        "train.focal_gamma": training_config.focal_gamma,
        "train.class_weight_power": training_config.class_weight_power,
        "train.label_smoothing": training_config.label_smoothing,
        "train.early_stopping": training_config.early_stopping,
        "train.patience": training_config.patience,
    })
    
    # Dataset params
    mlflow.log_params({
        "data.period_hours": dataset_config["period_hours"],
        "data.signal_shift": dataset_config["signal_shift"],
        "data.threshold_pct": dataset_config["threshold_pct"],
        "data.num_coins": len(symbols),
    })
    
    # Sequence params
    mlflow.log_params({
        "seq.input_length": sequence_config["input_seq_length"],
        "seq.output_length": sequence_config["output_seq_length"],
        "seq.stride": sequence_config["stride"],
    })
    
    # Distribution info
    train_dist = distributions['train']
    mlflow.log_params({
        "dist.train_samples": train_dist['total'],
        "dist.train_trade_pct": round(train_dist['trade_pct'], 2),
        "dist.imbalance_ratio": round(train_dist['imbalance_ratio'], 2),
    })


def log_training_metrics(history: TrainingHistory, epoch: int):
    """
    Log training metrics for a single epoch.
    """
    mlflow.log_metrics({
        "train_loss": history.train_losses[-1],
        "val_loss": history.val_losses[-1],
        "train_accuracy": history.train_accuracies[-1],
        "val_accuracy": history.val_accuracies[-1],
        "learning_rate": history.learning_rates[-1],
    }, step=epoch)


def log_evaluation_results(results: Dict[str, Dict]):
    """
    Log final evaluation metrics.
    """
    for split_name, metrics in results.items():
        if metrics is None:
            continue
        prefix = f"{split_name}_"
        mlflow.log_metrics({
            f"{prefix}accuracy": metrics['accuracy'],
            f"{prefix}precision": metrics['precision'],
            f"{prefix}recall": metrics['recall'],
            f"{prefix}f1": metrics['f1'],
            f"{prefix}hold_f1": metrics['hold_f1'],
            f"{prefix}trade_f1": metrics['trade_f1'],
        })


def log_model_artifact(model: torch.nn.Module, preprocessor: DataPreprocessor, output_dir: Path):
    """
    Log model and preprocessor as MLflow artifacts.
    """
    # Log PyTorch model
    mlflow.pytorch.log_model(model, "model")
    
    # Log preprocessor
    preprocessor_path = output_dir / "preprocessor_mlflow.pkl"
    with open(preprocessor_path, 'wb') as f:
        pickle.dump(preprocessor, f)
    mlflow.log_artifact(str(preprocessor_path))
    preprocessor_path.unlink()  # Clean up temp file

In [ ]:
# Cell 23: Setup MLflow experiment

experiment_id = setup_mlflow(MLFLOW_EXPERIMENT)

## Section 9: Training Loop

In [ ]:
# Cell 24: Training Function with MLflow

def train_with_mlflow(
    model: torch.nn.Module,
    training_config: TrainingConfig,
    train_dataset: SignalDataset,
    val_dataset: SignalDataset,
    test_dataset: SignalDataset,
    model_config: ModelConfig,
    dataset_config: Dict,
    sequence_config: Dict,
    symbols: List[str],
    distributions: Dict[str, Dict],
    preprocessor: DataPreprocessor,
    output_dir: Path
) -> Tuple[TrainingHistory, Dict, Trainer, str]:
    """
    Train model with full MLflow tracking.
    
    Returns
    -------
    Tuple[TrainingHistory, Dict, Trainer, str]
        (history, evaluation_results, trainer, run_id)
    """
    with mlflow.start_run() as run:
        run_id = run.info.run_id
        print(f"MLflow Run ID: {run_id}")
        print("=" * 60)
        
        # Log all parameters
        log_training_params(
            model_config, training_config, 
            dataset_config, sequence_config, 
            symbols, distributions
        )
        
        # Create trainer
        trainer = Trainer(
            model=model,
            config=training_config,
            preprocessor=preprocessor
        )
        
        # Store test dataset for evaluation
        trainer.test_dataset = test_dataset
        
        # Define callback for epoch-level logging
        def mlflow_callback(epoch: int, history: TrainingHistory):
            log_training_metrics(history, epoch)
        
        # Train
        print("\nStarting training...")
        history = trainer.train(
            train_dataset=train_dataset,
            val_dataset=val_dataset,
            callbacks=[mlflow_callback]
        )
        
        # Evaluate
        print("\nEvaluating model...")
        results = trainer.evaluate_all(verbose=True)
        
        # Log evaluation metrics
        log_evaluation_results(results)
        
        # Log final metrics
        mlflow.log_metrics({
            "best_epoch": history.best_epoch + 1,
            "best_val_loss": history.best_val_loss,
            "epochs_trained": len(history.train_losses),
        })
        
        # Log model artifact
        log_model_artifact(model, preprocessor, output_dir)
        
        print(f"\nMLflow Run completed: {run_id}")
        
    return history, results, trainer, run_id

In [ ]:
# Cell 25: Execute Training

# Get a reference preprocessor (all should be equivalent)
reference_preprocessor = list(processed_data.values())[0][2]

# Execute training with MLflow
history, eval_results, trainer, run_id = train_with_mlflow(
    model=model,
    training_config=training_config,
    train_dataset=multi_coin_datasets['train'],
    val_dataset=multi_coin_datasets['val'],
    test_dataset=multi_coin_datasets['test'],
    model_config=model_config,
    dataset_config=DATASET_CONFIG,
    sequence_config=SEQUENCE_CONFIG,
    symbols=symbols,
    distributions=dataset_distributions,
    preprocessor=reference_preprocessor,
    output_dir=OUTPUT_DIR
)

print(f"\nTraining complete!")
print(f"Best epoch: {history.best_epoch + 1}")
print(f"Best validation loss: {history.best_val_loss:.4f}")
print(f"MLflow Run ID: {run_id}")

## Section 10: Evaluation

In [ ]:
# Cell 26: Detailed Evaluation Report

# Print comprehensive evaluation report
trainer.print_evaluation_report(eval_results)

## Section 11: Visualization and Plots

In [ ]:
# Cell 27: Training Curves Plot

def plot_training_curves(
    history: TrainingHistory,
    output_path: Optional[Path] = None
) -> None:
    """
    Create and optionally save training curves visualization.
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    epochs = range(1, len(history.train_losses) + 1)
    
    # Loss curves
    ax = axes[0, 0]
    ax.plot(epochs, history.train_losses, 'b-', label='Train Loss', linewidth=2)
    ax.plot(epochs, history.val_losses, 'r-', label='Val Loss', linewidth=2)
    ax.axvline(x=history.best_epoch + 1, color='g', linestyle='--', 
               label=f'Best Epoch ({history.best_epoch + 1})', alpha=0.7)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Training & Validation Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Accuracy curves
    ax = axes[0, 1]
    ax.plot(epochs, history.train_accuracies, 'b-', label='Train Acc', linewidth=2)
    ax.plot(epochs, history.val_accuracies, 'r-', label='Val Acc', linewidth=2)
    ax.axvline(x=history.best_epoch + 1, color='g', linestyle='--', alpha=0.7)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.set_title('Training & Validation Accuracy')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Learning rate
    ax = axes[1, 0]
    ax.plot(epochs, history.learning_rates, 'purple', linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Learning Rate')
    ax.set_title('Learning Rate Schedule')
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3)
    
    # Loss difference (overfitting indicator)
    ax = axes[1, 1]
    loss_diff = np.array(history.val_losses) - np.array(history.train_losses)
    ax.plot(epochs, loss_diff, 'orange', linewidth=2)
    ax.axhline(y=0, color='black', linestyle='--', alpha=0.5)
    ax.fill_between(epochs, loss_diff, 0, where=(loss_diff > 0), 
                    alpha=0.3, color='red', label='Overfitting')
    ax.fill_between(epochs, loss_diff, 0, where=(loss_diff <= 0), 
                    alpha=0.3, color='green', label='Underfitting')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Val Loss - Train Loss')
    ax.set_title('Overfitting Indicator')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {output_path}")
    
    plt.show()

# Generate training curves
plot_training_curves(history, OUTPUT_DIR / "training_curves.png")

In [ ]:
# Cell 28: Confusion Matrices

def plot_confusion_matrices(
    results: Dict[str, Dict],
    output_path: Optional[Path] = None
) -> None:
    """
    Plot confusion matrices for all dataset splits.
    """
    splits = [k for k, v in results.items() if v is not None]
    n_splits = len(splits)
    
    fig, axes = plt.subplots(1, n_splits, figsize=(5 * n_splits, 4))
    if n_splits == 1:
        axes = [axes]
    
    for idx, split_name in enumerate(splits):
        metrics = results[split_name]
        cm = np.array(metrics['confusion_matrix'])
        
        sns.heatmap(
            cm,
            annot=True,
            fmt='d',
            cmap='Blues',
            xticklabels=['Hold', 'Trade'],
            yticklabels=['Hold', 'Trade'],
            ax=axes[idx]
        )
        axes[idx].set_xlabel('Predicted')
        axes[idx].set_ylabel('Actual')
        axes[idx].set_title(f'{split_name.upper()}\nF1: {metrics["f1"]:.3f}, '
                           f'Trade F1: {metrics["trade_f1"]:.3f}')
    
    plt.tight_layout()
    
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {output_path}")
    
    plt.show()

plot_confusion_matrices(eval_results, OUTPUT_DIR / "confusion_matrices.png")

In [ ]:
# Cell 29: Per-Coin Performance Analysis

def analyze_per_coin_performance(
    model: torch.nn.Module,
    split_data: Dict[str, Dict[str, Tuple[np.ndarray, np.ndarray]]],
    device: torch.device
) -> pd.DataFrame:
    """
    Analyze model performance on each coin separately (test set only).
    """
    from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
    
    model.eval()
    results = []
    
    for symbol, splits in split_data.items():
        feat_test, tgt_test = splits['test']
        
        if len(feat_test) == 0:
            continue
        
        # Create dataset and get predictions
        dataset = SignalDataset(feat_test, tgt_test, device=device)
        
        with torch.no_grad():
            features = dataset.features
            logits = model(features)
            preds = torch.argmax(logits, dim=-1).cpu().numpy()
            targets = dataset.targets.cpu().numpy()
        
        # Calculate metrics
        results.append({
            'Symbol': symbol,
            'Test Samples': len(targets),
            'Accuracy': accuracy_score(targets, preds),
            'F1': f1_score(targets, preds, average='weighted', zero_division=0),
            'Trade F1': f1_score(targets, preds, pos_label=1, zero_division=0),
            'Trade Precision': precision_score(targets, preds, pos_label=1, zero_division=0),
            'Trade Recall': recall_score(targets, preds, pos_label=1, zero_division=0),
            'Trade %': (targets == 1).mean() * 100
        })
    
    df = pd.DataFrame(results)
    df = df.sort_values('Trade F1', ascending=False)
    
    print("=" * 100)
    print("PER-COIN TEST PERFORMANCE")
    print("=" * 100)
    print(df.to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else x))
    print("=" * 100)
    
    return df

# Analyze per-coin performance
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
per_coin_results = analyze_per_coin_performance(model, split_coin_data, device)

In [ ]:
# Cell 30: Per-Coin Performance Visualization

def plot_per_coin_performance(
    per_coin_df: pd.DataFrame,
    output_path: Optional[Path] = None
) -> None:
    """
    Visualize per-coin performance metrics.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Sort by Trade F1
    df_sorted = per_coin_df.sort_values('Trade F1', ascending=True)
    
    # Trade F1 by coin
    ax = axes[0]
    colors = plt.cm.RdYlGn(df_sorted['Trade F1'].values)
    ax.barh(df_sorted['Symbol'], df_sorted['Trade F1'], color=colors)
    ax.set_xlabel('Trade F1 Score')
    ax.set_title('Trade F1 Score by Coin')
    ax.axvline(x=df_sorted['Trade F1'].mean(), color='red', linestyle='--', 
               label=f'Mean: {df_sorted["Trade F1"].mean():.3f}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Accuracy vs Trade F1 scatter
    ax = axes[1]
    scatter = ax.scatter(
        per_coin_df['Accuracy'], 
        per_coin_df['Trade F1'],
        c=per_coin_df['Trade %'],
        cmap='viridis',
        s=100,
        alpha=0.7
    )
    for idx, row in per_coin_df.iterrows():
        ax.annotate(row['Symbol'], (row['Accuracy'], row['Trade F1']), 
                   fontsize=8, ha='center', va='bottom')
    ax.set_xlabel('Accuracy')
    ax.set_ylabel('Trade F1')
    ax.set_title('Accuracy vs Trade F1 (color = Trade %)')
    plt.colorbar(scatter, ax=ax, label='Trade %')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {output_path}")
    
    plt.show()

plot_per_coin_performance(per_coin_results, OUTPUT_DIR / "per_coin_performance.png")

## Section 12: Summary Statistics

In [ ]:
# Cell 31: Experiment Summary Dataclass

@dataclass
class ExperimentSummary:
    """Summary of experiment for comparison."""
    timestamp: str
    run_id: str
    
    # Data info
    num_coins: int
    total_train_samples: int
    total_val_samples: int
    total_test_samples: int
    train_trade_pct: float
    imbalance_ratio: float
    
    # Model info
    model_type: str
    num_parameters: int
    hidden_size: int
    num_layers: int
    cnn_num_layers: int
    
    # Training info
    epochs_trained: int
    best_epoch: int
    best_val_loss: float
    learning_rate: float
    focal_loss: bool
    focal_gamma: float
    class_weight_power: float
    label_smoothing: float
    
    # Results
    train_accuracy: float
    val_accuracy: float
    test_accuracy: float
    train_f1: float
    val_f1: float
    test_f1: float
    test_trade_f1: float
    test_trade_precision: float
    test_trade_recall: float
    
    def to_dict(self) -> Dict:
        return asdict(self)


def create_experiment_summary(
    history: TrainingHistory,
    eval_results: Dict[str, Dict],
    model: torch.nn.Module,
    datasets: Dict[str, SignalDataset],
    training_config: TrainingConfig,
    model_config: ModelConfig,
    distributions: Dict[str, Dict],
    symbols: List[str],
    run_id: str
) -> ExperimentSummary:
    """
    Create experiment summary for tracking and comparison.
    """
    train_dist = distributions['train']
    
    summary = ExperimentSummary(
        timestamp=datetime.now().isoformat(),
        run_id=run_id,
        
        num_coins=len(symbols),
        total_train_samples=len(datasets['train']),
        total_val_samples=len(datasets['val']),
        total_test_samples=len(datasets['test']),
        train_trade_pct=round(train_dist['trade_pct'], 2),
        imbalance_ratio=round(train_dist['imbalance_ratio'], 2),
        
        model_type=model.__class__.__name__,
        num_parameters=model.get_num_parameters(),
        hidden_size=model_config.hidden_size,
        num_layers=model_config.num_layers,
        cnn_num_layers=model_config.cnn_num_layers,
        
        epochs_trained=len(history.train_losses),
        best_epoch=history.best_epoch + 1,
        best_val_loss=round(history.best_val_loss, 4),
        learning_rate=training_config.learning_rate,
        focal_loss=training_config.focal_loss,
        focal_gamma=training_config.focal_gamma,
        class_weight_power=training_config.class_weight_power,
        label_smoothing=training_config.label_smoothing,
        
        train_accuracy=round(eval_results['train']['accuracy'], 4),
        val_accuracy=round(eval_results['val']['accuracy'], 4),
        test_accuracy=round(eval_results['test']['accuracy'], 4) if eval_results.get('test') else 0,
        train_f1=round(eval_results['train']['f1'], 4),
        val_f1=round(eval_results['val']['f1'], 4),
        test_f1=round(eval_results['test']['f1'], 4) if eval_results.get('test') else 0,
        test_trade_f1=round(eval_results['test']['trade_f1'], 4) if eval_results.get('test') else 0,
        test_trade_precision=round(eval_results['test']['trade_precision'], 4) if eval_results.get('test') else 0,
        test_trade_recall=round(eval_results['test']['trade_recall'], 4) if eval_results.get('test') else 0,
    )
    
    return summary

In [ ]:
# Cell 32: Create and Display Experiment Summary

summary = create_experiment_summary(
    history, eval_results, model, multi_coin_datasets,
    training_config, model_config, dataset_distributions,
    symbols, run_id
)

print("\n" + "=" * 70)
print("EXPERIMENT SUMMARY")
print("=" * 70)

summary_dict = summary.to_dict()
sections = {
    'General': ['timestamp', 'run_id'],
    'Data': ['num_coins', 'total_train_samples', 'total_val_samples', 'total_test_samples', 
             'train_trade_pct', 'imbalance_ratio'],
    'Model': ['model_type', 'num_parameters', 'hidden_size', 'num_layers', 'cnn_num_layers'],
    'Training': ['epochs_trained', 'best_epoch', 'best_val_loss', 'learning_rate',
                 'focal_loss', 'focal_gamma', 'class_weight_power', 'label_smoothing'],
    'Results': ['train_accuracy', 'val_accuracy', 'test_accuracy', 'train_f1', 'val_f1',
                'test_f1', 'test_trade_f1', 'test_trade_precision', 'test_trade_recall']
}

for section_name, keys in sections.items():
    print(f"\n{section_name}:")
    for key in keys:
        value = summary_dict[key]
        if isinstance(value, float) and not isinstance(value, bool):
            print(f"  {key}: {value:.4f}")
        else:
            print(f"  {key}: {value}")

print("\n" + "=" * 70)

In [ ]:
# Cell 33: Save Experiment Summary

def save_experiment_summary(
    summary: ExperimentSummary,
    output_dir: Path
) -> Path:
    """
    Save experiment summary to JSON.
    """
    timestamp_str = datetime.now().strftime('%Y%m%d_%H%M%S')
    summary_path = output_dir / f"experiment_summary_{timestamp_str}.json"
    
    with open(summary_path, 'w') as f:
        json.dump(summary.to_dict(), f, indent=2)
    
    print(f"Saved summary: {summary_path}")
    return summary_path

# Save the summary
summary_path = save_experiment_summary(summary, OUTPUT_DIR)

In [ ]:
# Cell 34: Per-Coin Results to CSV

# Save per-coin results
per_coin_path = OUTPUT_DIR / "per_coin_test_results.csv"
per_coin_results.to_csv(per_coin_path, index=False)
print(f"Saved per-coin results: {per_coin_path}")

## Next Steps

1. **View MLflow UI**: Run `mlflow ui` in terminal to view experiment tracking
2. **Compare Experiments**: Modify configurations and re-run to compare results
3. **Hyperparameter Tuning**: Adjust model and training configs based on results
4. **Production**: Use best model for trading signal predictions

In [ ]:
# Cell 35: Final Summary - Files Created

print("\n" + "=" * 70)
print("FILES CREATED")
print("=" * 70)

# List all created files
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        size_kb = path.stat().st_size / 1024
        print(f"  {path.relative_to(OUTPUT_DIR)}: {size_kb:.1f} KB")

print("\n" + "=" * 70)
print("To view MLflow experiments, run in terminal:")
print("  cd notebooks && mlflow ui")
print("Then open http://localhost:5000 in your browser")
print("=" * 70)